In [ ]:
!pip -q install python-pptx pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 19.6 MB/s eta 0:00:00


Imports + Helpers

In [ ]:
import os, re, json, glob
import pandas as pd
from pptx import Presentation

# Regex principales
RE_SEM = re.compile(r"\bS(5|6|7|8|9|10)\b", re.IGNORECASE)
RE_TYPE = re.compile(r"\b(TP|PROJETS?|PROJET|STAGE|AUTRES?)\b", re.IGNORECASE)
RE_COMP = re.compile(r"\bC\d(?:-\d)?\b", re.IGNORECASE)  # C1, C2, C4-1...

def norm_space(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip())

def infer_specialite_from_filename(path: str) -> str:
    base = os.path.basename(path)
    base = os.path.splitext(base)[0]
    return norm_space(base)


Charger les fichiers PPTX

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PPTX_DIR = "/content/drive/MyDrive/cartographies"
pptx_paths = glob.glob(os.path.join(PPTX_DIR, "*.pptx"))
pptx_paths


Mounted at /content/drive


['/content/drive/MyDrive/cartographies/Cartographie MECA - MC.pptx',
 '/content/drive/MyDrive/cartographies/Cartographie BEE FISA_V2.pptx',
 '/content/drive/MyDrive/cartographies/Cartographie MECA - GI- v2.pptx',
 '/content/drive/MyDrive/cartographies/Cartographie MECA - GI.pptx',
 '/content/drive/MyDrive/cartographies/Cartographie MECA - CM.pptx',
 '/content/drive/MyDrive/cartographies/Cartographie MECA - MI .pptx',
 '/content/drive/MyDrive/cartographies/Cartographie BEE__FISE_v2.pptx',
 '/content/drive/MyDrive/cartographies/Cartographie SNI.pptx',
 '/content/drive/MyDrive/cartographies/Cartographie EIT.pptx',
 '/content/drive/MyDrive/cartographies/Cartographie IDU.pptx']

Extraction brute (texte + positions)

In [ ]:
def extract_raw_blocks(pptx_path: str):
    prs = Presentation(pptx_path)
    blocks = []
    for si, slide in enumerate(prs.slides, start=1):
        for shape in slide.shapes:
            # Texte
            if getattr(shape, "has_text_frame", False) and shape.has_text_frame:
                text = norm_space(shape.text)
                if not text:
                    continue
                blocks.append({
                    "source_file": os.path.basename(pptx_path),
                    "specialite": infer_specialite_from_filename(pptx_path),
                    "slide_number": si,
                    "text": text,
                    "left": int(shape.left),
                    "top": int(shape.top),
                    "width": int(shape.width),
                    "height": int(shape.height),
                })
    return blocks


Structuration “APC” (heuristique robuste)

In [ ]:
def build_structured_from_blocks(blocks):
    """
    Retourne:
      activities: liste de dicts (activité structurée)
      links: liste de dicts (activité_id -> compétence)
    """
    # Index interne
    activities = []
    links = []
    activity_id = 0

    # Groupement par fichier + slide
    df = pd.DataFrame(blocks)
    if df.empty:
        return activities, links

    for (source_file, slide_number), g in df.groupby(["source_file", "slide_number"]):
        g = g.copy()

        # Trier pour une lecture "visuelle" (haut->bas puis gauche->droite)
        g = g.sort_values(["top", "left"])

        # Repérer en-têtes de semestres
        sem_headers = []
        for _, row in g.iterrows():
            m = RE_SEM.search(row["text"])
            if m and len(row["text"]) <= 6:  # ex: "S5"
                sem_headers.append((m.group(0).upper(), row["left"], row["top"]))
        # Si pas d'en-têtes courts, on repère quand même S5..S10 dans le texte
        # (fallback)
        if not sem_headers:
            for _, row in g.iterrows():
                m = RE_SEM.search(row["text"])
                if m:
                    sem_headers.append((m.group(0).upper(), row["left"], row["top"]))

        # Repérer zones de type (TP/PROJET/STAGE/AUTRES)
        type_headers = []
        for _, row in g.iterrows():
            m = RE_TYPE.search(row["text"])
            if m and len(row["text"]) <= 12:
                t = m.group(0).upper()
                # normalisation
                if t.startswith("PROJET"):
                    t = "PROJETS"
                if t.startswith("AUTRE"):
                    t = "AUTRES"
                type_headers.append((t, row["left"], row["top"]))

        # Fonctions d'affectation
        def nearest_sem(left):
            if not sem_headers:
                return None
            return min(sem_headers, key=lambda s: abs(left - s[1]))[0]

        def nearest_type(top):
            if not type_headers:
                return None
            return min(type_headers, key=lambda t: abs(top - t[2]))[0]

        # Parcours des blocs: identifier activités candidates
        # Heuristique: une activité est un bloc texte qui n'est pas un header (Sx, TP/PROJETS/STAGE/AUTRES)
        # et qui n'est pas "légende" (si vous avez une légende, elle a souvent des mots fixes)
        for _, row in g.iterrows():
            text = row["text"]

            # ignorer headers purs
            if RE_SEM.fullmatch(text.upper()):
                continue
            if RE_TYPE.fullmatch(text.upper()):
                continue

            # ignorer légende (adaptable)
            if "LÉGENDE" in text.upper() or "LEGENDE" in text.upper():
                continue

            # Extraire compétences du bloc lui-même
            comps_in_text = sorted(set([c.upper() for c in RE_COMP.findall(text)]))

            # Nettoyage simple: enlever les compétences du titre si elles sont concaténées
            cleaned_name = norm_space(RE_COMP.sub("", text))

            sem = nearest_sem(row["left"])
            typ = nearest_type(row["top"])

            # Filtre minimal : garder les blocs qui ressemblent à une activité
            # (au moins 3 caractères, pas juste "C1", etc.)
            if len(cleaned_name) < 3:
                continue

            activity_id += 1
            act = {
                "activity_id": activity_id,
                "source_file": source_file,
                "specialite": row["specialite"],
                "slide_number": int(slide_number),
                "semester": sem,
                "type": typ,
                "name": cleaned_name,
                "raw_text": text,
                "left": int(row["left"]),
                "top": int(row["top"]),
            }
            activities.append(act)

            for c in comps_in_text:
                links.append({
                    "activity_id": activity_id,
                    "competence": c
                })

        # (Optionnel) Association par proximité :
        # Si les compétences sont dans des blocs séparés (ex: une ligne "C1 C2"),
        # on les rattache à l’activité la plus proche au-dessus dans la même "colonne".
        # Activation simple :
        comp_rows = []
        for _, row in g.iterrows():
            comps = sorted(set([c.upper() for c in RE_COMP.findall(row["text"])]))
            if comps and len(norm_space(RE_COMP.sub("", row["text"]))) <= 2:
                comp_rows.append((row, comps))

        if comp_rows and activities:
            # activités du slide
            acts_slide = [a for a in activities if a["source_file"] == source_file and a["slide_number"] == int(slide_number)]
            for row, comps in comp_rows:
                # activité la plus proche au-dessus (diff top positive minimale) + proche en X
                candidates = []
                for a in acts_slide:
                    dy = row["top"] - a["top"]
                    dx = abs(row["left"] - a["left"])
                    if dy > 0 and dy < 220:  # fenêtre verticale (à ajuster si besoin)
                        candidates.append((dy, dx, a["activity_id"]))
                if candidates:
                    _, _, aid = min(candidates, key=lambda x: (x[0], x[1]))
                    for c in comps:
                        links.append({"activity_id": aid, "competence": c})

    # Dé-doublonnage links
    if links:
        links = pd.DataFrame(links).drop_duplicates().to_dict(orient="records")

    return activities, links


Batch : traiter tous les PPTX + exporter CSV & JSON

In [ ]:
OUT_DIR = "exports_apc"
os.makedirs(OUT_DIR, exist_ok=True)

all_raw = []
all_activities = []
all_links = []

for path in pptx_paths:
    raw = extract_raw_blocks(path)
    acts, links = build_structured_from_blocks(raw)

    all_raw.extend(raw)
    all_activities.extend(acts)
    all_links.extend(links)

    # Exports par fichier
    base = os.path.splitext(os.path.basename(path))[0]

    # RAW
    pd.DataFrame(raw).to_csv(os.path.join(OUT_DIR, f"{base}_raw.csv"), index=False, encoding="utf-8")
    with open(os.path.join(OUT_DIR, f"{base}_raw.json"), "w", encoding="utf-8") as f:
        json.dump(raw, f, ensure_ascii=False, indent=2)

    # STRUCT
    pd.DataFrame(acts).to_csv(os.path.join(OUT_DIR, f"{base}_activities.csv"), index=False, encoding="utf-8")
    with open(os.path.join(OUT_DIR, f"{base}_activities.json"), "w", encoding="utf-8") as f:
        json.dump(acts, f, ensure_ascii=False, indent=2)

    pd.DataFrame(links).to_csv(os.path.join(OUT_DIR, f"{base}_activity_competence.csv"), index=False, encoding="utf-8")
    with open(os.path.join(OUT_DIR, f"{base}_activity_competence.json"), "w", encoding="utf-8") as f:
        json.dump(links, f, ensure_ascii=False, indent=2)

# Export global
pd.DataFrame(all_raw).to_csv(os.path.join(OUT_DIR, "ALL_raw.csv"), index=False, encoding="utf-8")
with open(os.path.join(OUT_DIR, "ALL_raw.json"), "w", encoding="utf-8") as f:
    json.dump(all_raw, f, ensure_ascii=False, indent=2)

pd.DataFrame(all_activities).to_csv(os.path.join(OUT_DIR, "ALL_activities.csv"), index=False, encoding="utf-8")
with open(os.path.join(OUT_DIR, "ALL_activities.json"), "w", encoding="utf-8") as f:
    json.dump(all_activities, f, ensure_ascii=False, indent=2)

pd.DataFrame(all_links).to_csv(os.path.join(OUT_DIR, "ALL_activity_competence.csv"), index=False, encoding="utf-8")
with open(os.path.join(OUT_DIR, "ALL_activity_competence.json"), "w", encoding="utf-8") as f:
    json.dump(all_links, f, ensure_ascii=False, indent=2)

print("✅ Exports générés dans:", OUT_DIR)


✅ Exports générés dans: exports_apc


Télécharger les résultats

In [ ]:
import shutil
archive_path = shutil.make_archive("exports_apc", "zip", OUT_DIR)

from google.colab import files
files.download(archive_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>